In [2]:
import pandas as pd
from pathlib import Path

DATA_PROCESSED = Path("data/processed")

sales = pd.read_csv(
    DATA_PROCESSED / "sales_clean.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "shipping_limit_date",
        "review_creation_date",
        "review_answer_timestamp",
        "order_delivered_carrier_date",
    ],
)

In [3]:
sales.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,13.0,9350.0,maua,SP,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,13.0,9350.0,maua,SP,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,13.0,9350.0,maua,SP,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,19.0,31570.0,belo horizonte,SP,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,21.0,14840.0,guariba,SP,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18,2018-08-22 19:07:58


In [4]:
sales["purchase_year"] = sales["order_purchase_timestamp"].dt.year

sales["purchase_month"] = sales["order_purchase_timestamp"].dt.month

sales["purchase_quarter"] = sales["order_purchase_timestamp"].dt.quarter

sales["purchase_day"] = sales["order_purchase_timestamp"].dt.day

sales["purchase_weekday"] = sales["order_purchase_timestamp"].dt.day_name()

sales["purchase_hour"] = sales["order_purchase_timestamp"].dt.hour

In [5]:
sales["delivery_time_days"] = (
    sales["order_delivered_customer_date"]
    - sales["order_purchase_timestamp"]
).dt.days

In [6]:
sales["delivery_delay_days"] = (
    sales["order_delivered_customer_date"]
    - sales["order_estimated_delivery_date"]
).dt.days

In [7]:
sales["approval_time_hours"] = (
    sales["order_approved_at"]
    - sales["order_purchase_timestamp"]
).dt.total_seconds() / 3600

In [8]:
sales["shipping_time_days"] = (
    sales["order_delivered_carrier_date"]
    - sales["order_approved_at"]
).dt.days

In [9]:
sales["total_order_value"] = (
    sales["price"] + sales["freight_value"]
)

In [10]:
sales["freight_ratio"] = (
    sales["freight_value"]
    / sales["total_order_value"]
)

In [12]:
import numpy as np
sales["review_label"] = np.where(
    sales["review_score"] >= 4,
    "Satisfied",
    "Unsatisfied"
)

In [13]:
sales["is_delayed"] = (
    sales["delivery_delay_days"] > 0
).astype(int)

In [14]:
items_per_order = (
    sales.groupby("order_id")
         .size()
         .rename("items_per_order")
)

In [15]:
sales = sales.merge(
    items_per_order,
    on="order_id",
    how="left"
)

In [16]:
sales[
    [
        "delivery_time_days",
        "delivery_delay_days",
        "approval_time_hours",
        "shipping_time_days",
        "total_order_value",
        "freight_ratio",
        "items_per_order",
        "review_label",
        "is_delayed",
    ]
].head()

,delivery_time_days,delivery_delay_days,approval_time_hours,shipping_time_days,total_order_value,freight_ratio,items_per_order,review_label,is_delayed
0,8.0,-8.0,0.178333,2.0,38.71,0.225265,3,Satisfied,0
1,8.0,-8.0,0.178333,2.0,38.71,0.225265,3,Satisfied,0
2,8.0,-8.0,0.178333,2.0,38.71,0.225265,3,Satisfied,0
3,13.0,-6.0,30.713889,0.0,141.46,0.160894,1,Satisfied,0
4,9.0,-18.0,0.276111,0.0,179.12,0.107302,1,Satisfied,0


In [17]:
sales.to_csv(
    DATA_PROCESSED / "sales_featured.csv",
    index=False
)